## Download the exercise data
Run the next cell once before starting the exercise. It downloads and extracts this notebook’s data into `~/kenya2026`. Set `KENYA2026_WORK_DIR` first if you prefer another location.

In [ ]:
from pathlib import Path
import os
import subprocess

exercise = "day4_morning_fst"
base_url = "https://popgen.dk/albrecht/course/kenya2026/data"
work_dir = Path(os.environ.get("KENYA2026_WORK_DIR", Path.home() / "kenya2026")).expanduser()
exercise_dir = work_dir / exercise
archive = work_dir / f"{exercise}.zip"
work_dir.mkdir(parents=True, exist_ok=True)

if not archive.exists():
    subprocess.run(["wget", "-c", f"{base_url}/{exercise}.zip", "-O", str(archive)], check=True)
if not exercise_dir.exists():
    subprocess.run(["unzip", "-q", str(archive), "-d", str(work_dir)], check=True)

os.chdir(exercise_dir)
print(f"Working directory: {Path.cwd()}")

### Software requirements
The setup cell above downloads **only the exercise data**. It does not install software. Before running the rest of this notebook, install the command-line programs and the Python or R packages that are imported or called in the exercises. If you see an error such as `command not found`, `ModuleNotFoundError`, or `there is no package called ...`, install the named dependency or ask an instructor for help.

# $F_{st}$ in wildebeest

We are back to wildebeest!

In this exercise we will cover:
 - Generating and displaying pairwise $F_{st}$ values
    
    
Tools used: plink2, R

The notebooks are editable, so feel free to experiment and change the code to see what happens or write notes in the text cells. Just remember to download the notebooks used here at some point if you want to save them with your own changes included.

In [ ]:
### make directory for the exercise
mkdir -p ~/kenya2026/Fst
cd ~/kenya2026/Fst

We will be using the data set of called genotypes from different blue wildebeest populations, as well as some black wildebeest as an outgroup to compare to, saved in a plink format file set. 

Here is the map from earlier to help show the sampling locations of the different wildebeest populations:
<img src="https://raw.githubusercontent.com/popgenDK/popgenDK.github.io/gh-pages/images/slider/wildeBeastMap.png" alt="image info" />


 **- Do you remember what a plink file set (.bed, bim and .fam) contains?**

In [ ]:
head wildebeest_fst.fam

In [ ]:
zstdcat wildebeest_fst.bim.zst | head

Below we have the command used to run the $F_{st}$ estimation:

In [ ]:
plink2 --bfile wildebeest_fst vzs --within clusterfile \
    --fst CATPHENO method=hudson --allow-extra-chr --threads 10

 **- How many individuals are in this files? And divided in how many populations?**

The cluster/within file supplied with the packaged exercise data tells the program how to separate the individuals into different groups for comparison. If we did not know up front which samples belonged together in populations, can you recall something we have looked at that could perhaps help with this?

Then let's have a look at the results:

In [ ]:
# some hartebeest samples were also originally included in this data set, but now we can just remove those from
# the output
grep -ve Hartebeest plink2.fst.summary > tmp
mv tmp plink2.fst.summary

# print the results
column -t plink2.fst.summary

 **- Which populations are most genetically differentiated? Which are most similar?**
 
 **- Can you indentify a pattern in the Fst values between black wildebeest and each of the blue wildebeest populations? Try to see if you can explain this pattern.**

Are each of these values large or small? This is quite difficult to answer without context, as it will depend on the type of data you are analyzing, the amount of data and the scope of your study. To provide context, one often looks at a matrix of $F_{st}$ values, which can be visualized using a heatmap. To do this we first need to transform the above data frame into a matrix, and then generate a heatmap using the heatmap.2-function.

In [ ]:
options(repr.matrix.max.cols=10, repr.matrix.max.rows=10)
options(repr.plot.width=16, repr.plot.height=16)
library(gplots)

# read the data into R
fst <- read.table("~/kenya2026/Fst/plink2.fst.summary")
names(fst) <- c("pop1", "pop2", "est")
fst <- fst[fst$pop1 != "Hartebeest" & fst$pop2 != "Hartebeest",]

Here we transform the table from above into a pairwise matrix that contains the exact same information, just in a different format:

In [ ]:
mat <- matrix(NA, 8, 8)
mat[lower.tri(mat)] <- fst$est
mat <- t(mat)
mat[lower.tri(mat)] <- fst$est
colnames(mat) <- c( "Amboseli", fst[1:7,2])
rownames(mat) <- c( "Amboseli", fst[1:7,2])
mat

In [ ]:
heatmap.2(mat, symm=T, trace='n', cexRow=1.5, cexCol=1.5, margins = c(12, 12))

**- Look at the clustering tree produced by this method. Do the different groups relate to each other as we would expect?**

**- We can see some discrete levels of values in the color key and in the histogram in the inset plot. What do these correspond to?**
 
An important note here is that the tree/dendrogram used to order the groups here simply comes from clustering based on the $F_{st}$ values and will not neccesarily reflect the true evolutionary history of the groups.

In [ ]:
# run to start quiz
from jupyterquiz import display_quiz
display_quiz('https://raw.githubusercontent.com/popgenDK/courses/main/kenya2026/exercises/Day4/quiz_wildebeest_fst.json')

# Extra/optional - Reindeer SAF-based $F_{ST}$ - Using genotype Likelihood

We now estimate genotype-likelihood-based differentiation for **Qassit**, **Neria**, and **Ameralik**. The ANGSD SAF files are supplied in `reindeer_saf`; do not regenerate them.

Each population needs a matching `.saf.idx`, `.saf.gz`, and `.saf.pos.gz` triplet. These SAFs must be full-dimensional (not generated with `ANGSD -fold 1`) and must use the same reference and compatible filters.

`winsfs` estimates the pairwise 2D-SFS. We then remove its `#SHAPE` header and use ANGSD's `realSFS fst` functions for global and windowed $F_{ST}$.

In [ ]:
mkdir -p ~/kenya2026/reindeer_saf_fst
cd ~/kenya2026/reindeer_saf_fst

SAF_DIR=reindeer_saf
pops=(Qassit Neria Ameralik)
extensions=(saf.idx saf.gz saf.pos.gz)
missing=0

for pop in "${pops[@]}"; do
  for extension in "${extensions[@]}"; do
    file="$SAF_DIR/$pop.$extension"
    if [[ ! -s "$file" ]]; then
      echo "MISSING: $file" >&2
      missing=1
    fi
  done
done

if (( missing )); then
  echo "Add all nine supplied SAF components to $SAF_DIR before continuing." >&2
  exit 1
fi

echo "All supplied SAF components were found."
ls -lh "$SAF_DIR"/{Qassit,Neria,Ameralik}.saf.{idx,gz,pos.gz}

In [ ]:
echo "This command was used to generate the precomputed files:"
echo "SAF_DIR=reindeer_saf
OUT=~/kenya2026/reindeer_saf_fst
THREADS=40

pairs=("Qassit Neria" "Qassit Ameralik" "Neria Ameralik")

for pair in "${pairs[@]}"; do
  read -r pop1 pop2 <<< "$pair"
  prefix="$OUT/${pop1}_${pop2}"
  echo "Estimating $pop1 versus $pop2"

  winsfs --threads "$THREADS" --seed 2026 \
    "$SAF_DIR/$pop1.saf.idx" "$SAF_DIR/$pop2.saf.idx" \
    > "$prefix.winsfs"

  # winsfs writes a #SHAPE header; realSFS fst expects only numeric SFS values.
  tail -n 1 "$prefix.winsfs" > "$prefix.2dsfs"

  realSFS fst index \
    "$SAF_DIR/$pop1.saf.idx" "$SAF_DIR/$pop2.saf.idx" \
    -sfs "$prefix.2dsfs" -fstout "$prefix"

  realSFS fst stats "$prefix.fst.idx" > "$prefix.global_fst.txt"
  realSFS fst stats2 "$prefix.fst.idx" -win 100000 -step 50000 \
    > "$prefix.100kb_fst.tsv"
done"

echo
echo "Estimating Qassit versus Neria
Estimating Qassit versus Ameralik
Estimating Neria versus Ameralik"

In [ ]:
echo ----Global pairwise Fst results----
for result in reindeer_saf/*.global_fst.txt; do
  echo "Population Pair  100kb    global"
  printf '%s: ' "$(basename "$result" .global_fst.txt)"
  cat "$result"
  echo
done

**Which population pair has the largest global $F_{ST}$? Which has the smallest?**

**Do particular 100-kb windows show much stronger differentiation than the global estimate?**

**Why must the three SAF datasets use compatible genomic sites, references, and filters?**

In [ ]:
# run to start quiz
from jupyterquiz import display_quiz
display_quiz('https://raw.githubusercontent.com/popgenDK/courses/main/kenya2026/exercises/Day4/quiz_reindeer_saf_fst.json')